# 2. Veri Ön İşleme ve Birleştirme

Bu notebook'ta verileri temizleyecek, yıldırım olaylarını en yakın istasyona eşleştirip tüm verileri birleştireceğiz.

**İçerik:**
- Önceki verilerin yüklenmesi
- Koordinat bazlı istasyon eşleştirmesi
- Günlük agregasyon
- Verilerin birleştirilmesi

In [1]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist
import warnings
import os

warnings.filterwarnings('ignore')
print('Kütüphaneler yüklendi!')

Kütüphaneler yüklendi!


## 2.1 Verilerin Yüklenmesi

In [2]:
# Önceki notebook'tan kaydedilen verileri yükle
DATA_PATH = '../data/'

yildirim_df = pd.read_pickle(DATA_PATH + 'yildirim_processed.pkl')
istasyonlar = pd.read_pickle(DATA_PATH + 'istasyonlar.pkl')

print(f"Yıldırım verisi: {len(yildirim_df):,} kayıt")
print(f"İstasyon sayısı: {len(istasyonlar)}")

Yıldırım verisi: 258,815 kayıt
İstasyon sayısı: 6


## 2.2 En Yakın İstasyon Eşleştirmesi

In [3]:
def en_yakin_istasyon(enlem, boylam, istasyonlar_df):
    """Verilen koordinata en yakın istasyonu bulur."""
    ist_coords = istasyonlar_df[['enlem', 'boylam']].values
    nokta = np.array([[enlem, boylam]])
    mesafeler = cdist(nokta, ist_coords, metric='euclidean')[0]
    en_yakin_idx = np.argmin(mesafeler)
    return istasyonlar_df.iloc[en_yakin_idx]['istasyon_no']

# Her yıldırım için en yakın istasyonu bul
print("En yakın istasyon eşleştirmesi yapılıyor...")
yildirim_df['istasyon_no'] = yildirim_df.apply(
    lambda row: en_yakin_istasyon(row['enlem'], row['boylam'], istasyonlar), axis=1
)
print("Eşleştirme tamamlandı!")

# İstasyon başına dağılım
print("\nİstasyon başına yıldırım dağılımı:")
print(yildirim_df['istasyon_no'].value_counts())

En yakın istasyon eşleştirmesi yapılıyor...
Eşleştirme tamamlandı!

İstasyon başına yıldırım dağılımı:
istasyon_no
18771    100498
17285     70515
17815     62597
19909     14603
18234      7080
17920      3522
Name: count, dtype: int64


## 2.3 Günlük Agregasyon (İstasyon Bazında)

In [4]:
# İstasyon ve tarih bazında günlük yıldırım sayısı
gunluk_ist_yildirim = yildirim_df.groupby(['tarih', 'istasyon_no']).agg({
    'akim_kA': ['count', 'mean', 'max', 'min'],
    'mesafe_km': 'mean'
}).reset_index()

# Sütun isimlerini düzelt
gunluk_ist_yildirim.columns = ['tarih', 'istasyon_no', 'yildirim_sayisi', 
                                'ort_akim', 'maks_akim', 'min_akim', 'ort_mesafe']

gunluk_ist_yildirim['tarih'] = pd.to_datetime(gunluk_ist_yildirim['tarih'])

print(f"Günlük istasyon bazlı kayıt sayısı: {len(gunluk_ist_yildirim):,}")
display(gunluk_ist_yildirim.head(10))

Günlük istasyon bazlı kayıt sayısı: 3,303


,tarih,istasyon_no,yildirim_sayisi,ort_akim,maks_akim,min_akim,ort_mesafe
0,2015-02-06,18771,2,-6.000000,-6,-6,35.500000
1,2015-02-09,17285,8,11.750000,85,-32,21.650000
2,2015-02-09,17815,8,-4.250000,26,-43,31.575000
3,2015-02-09,18234,2,22.000000,22,22,47.800000
4,2015-02-09,18771,18,0.888889,30,-69,43.644444
5,2015-02-09,19909,2,23.000000,23,23,13.600000
6,2015-02-10,17285,14,-6.285714,71,-94,27.385714
7,2015-02-10,17815,14,-9.285714,37,-56,33.600000
8,2015-02-10,17920,2,31.000000,31,31,46.600000
9,2015-02-10,18234,2,18.000000,18,18,49.300000


## 2.4 Meteorolojik Verilerin Düzenlenmesi

In [5]:
def duzenle_meteo(dosya_adi):
    """Meteorolojik veriyi long format'a çevirir."""
    try:
        df = pd.read_pickle(DATA_PATH + dosya_adi)
        
        # İlk sütun tarih, diğerleri istasyonlar olmalı
        # Sütun yapısını incele
        print(f"\n{dosya_adi} sütunları: {df.columns.tolist()[:5]}...")
        
        return df
    except Exception as e:
        print(f"Hata ({dosya_adi}): {e}")
        return None

# Meteorolojik verileri yükle
meteo_files = [f for f in os.listdir(DATA_PATH) if f.startswith('meteo_')]
print("Meteorolojik veri dosyaları:")
for f in meteo_files:
    print(f"  - {f}")

Meteorolojik veri dosyaları:
  - meteo_basinc.pkl
  - meteo_bulut.pkl
  - meteo_maks_sicaklik.pkl
  - meteo_min_sicaklik.pkl
  - meteo_nem.pkl
  - meteo_ort_sicaklik.pkl
  - meteo_ruzgar.pkl
  - meteo_yagis.pkl


In [6]:
# Her meteorolojik veriyi yükle ve incele
meteo_dict = {}
for f in meteo_files:
    veri_adi = f.replace('meteo_', '').replace('.pkl', '')
    meteo_dict[veri_adi] = pd.read_pickle(DATA_PATH + f)
    print(f"{veri_adi}: {meteo_dict[veri_adi].shape}")

basinc: (14432, 6)
bulut: (7540, 6)
maks_sicaklik: (19032, 6)
min_sicaklik: (19037, 6)
nem: (20167, 6)
ort_sicaklik: (20200, 6)
ruzgar: (17032, 6)
yagis: (7840, 6)


In [7]:
# Örnek bir meteorolojik veri yapısını incele
ornek_veri = list(meteo_dict.values())[0]
print("Örnek veri yapısı:")
display(ornek_veri.head())
print(f"\nSütunlar: {ornek_veri.columns.tolist()}")

Örnek veri yapısı:


,Istasyon_No,Istasyon_Adi,YIL,AY,GUN,ORTALAMA_AKTUEL_BASINC_hPa
0,17285,HAKKARİ,2015.0,3.0,1.0,824.3
1,17285,HAKKARİ,2015.0,3.0,2.0,823.0
2,17285,HAKKARİ,2015.0,3.0,3.0,823.6
3,17285,HAKKARİ,2015.0,3.0,4.0,822.9
4,17285,HAKKARİ,2015.0,3.0,5.0,824.5



Sütunlar: ['Istasyon_No', 'Istasyon_Adi', 'YIL', 'AY', 'GUN', 'ORTALAMA_AKTUEL_BASINC_hPa']


In [8]:
def melted_meteo(df, deger_adi):
    """Meteorolojik veriyi long format'a çevirir."""
    # İlk sütunun tarih olduğunu varsayalım
    tarih_sutun = df.columns[0]
    
    # Melt işlemi
    df_melted = df.melt(id_vars=[tarih_sutun], var_name='istasyon_info', value_name=deger_adi)
    
    # Tarih sütununu datetime'a çevir
    df_melted['tarih'] = pd.to_datetime(df_melted[tarih_sutun], errors='coerce')
    
    return df_melted[['tarih', 'istasyon_info', deger_adi]]

# Test
test_melted = melted_meteo(meteo_dict['maks_sicaklik'], 'maks_sicaklik')
print("Melted veri:")
display(test_melted.head(10))

Melted veri:


,tarih,istasyon_info,maks_sicaklik
0,NaT,Istasyon_Adi,HAKKARİ
1,NaT,Istasyon_Adi,HAKKARİ
2,NaT,Istasyon_Adi,HAKKARİ
3,NaT,Istasyon_Adi,HAKKARİ
4,NaT,Istasyon_Adi,HAKKARİ
5,NaT,Istasyon_Adi,HAKKARİ
6,NaT,Istasyon_Adi,HAKKARİ
7,NaT,Istasyon_Adi,HAKKARİ
8,NaT,Istasyon_Adi,HAKKARİ
9,NaT,Istasyon_Adi,HAKKARİ


## 2.5 Tam Tarih Aralığı Oluşturma

In [9]:
# Veri setinin tarih aralığını belirle
min_tarih = yildirim_df['tarih'].min()
max_tarih = yildirim_df['tarih'].max()

print(f"Tarih aralığı: {min_tarih} - {max_tarih}")

# Tüm tarihler için DataFrame oluştur
tum_tarihler = pd.date_range(start=min_tarih, end=max_tarih, freq='D')
print(f"Toplam gün sayısı: {len(tum_tarihler)}")

# Her istasyon için tüm tarihleri içeren DataFrame
from itertools import product

tum_kombinasyonlar = pd.DataFrame(
    list(product(tum_tarihler, istasyonlar['istasyon_no'])),
    columns=['tarih', 'istasyon_no']
)

print(f"Toplam kombinasyon: {len(tum_kombinasyonlar):,}")

Tarih aralığı: 2015-02-06 - 2025-11-16
Toplam gün sayısı: 3937
Toplam kombinasyon: 23,622


In [10]:
# Yıldırım verisiyle birleştir
ana_df = tum_kombinasyonlar.merge(
    gunluk_ist_yildirim,
    on=['tarih', 'istasyon_no'],
    how='left'
)

# NaN değerleri 0 ile doldur (yıldırım olmayan günler)
ana_df['yildirim_sayisi'] = ana_df['yildirim_sayisi'].fillna(0).astype(int)

# Hedef değişken: Yıldırım var mı yok mu (binary)
ana_df['yildirim_var'] = (ana_df['yildirim_sayisi'] > 0).astype(int)

print(f"Ana veri seti boyutu: {ana_df.shape}")
print(f"\nHedef değişken dağılımı:")
print(ana_df['yildirim_var'].value_counts())
print(f"\nYıldırım olan gün oranı: {ana_df['yildirim_var'].mean()*100:.2f}%")

Ana veri seti boyutu: (23622, 8)

Hedef değişken dağılımı:
yildirim_var
0    20319
1     3303
Name: count, dtype: int64

Yıldırım olan gün oranı: 13.98%


## 2.6 Zaman Özellikleri Ekleme

In [11]:
# Zaman özellikleri
ana_df['yil'] = ana_df['tarih'].dt.year
ana_df['ay'] = ana_df['tarih'].dt.month
ana_df['gun'] = ana_df['tarih'].dt.day
ana_df['haftanin_gunu'] = ana_df['tarih'].dt.dayofweek
ana_df['yilin_gunu'] = ana_df['tarih'].dt.dayofyear

# Mevsim
def mevsim_bul(ay):
    if ay in [12, 1, 2]:
        return 0  # Kış
    elif ay in [3, 4, 5]:
        return 1  # İlkbahar
    elif ay in [6, 7, 8]:
        return 2  # Yaz
    else:
        return 3  # Sonbahar

ana_df['mevsim'] = ana_df['ay'].apply(mevsim_bul)

print("Zaman özellikleri eklendi!")
display(ana_df.head())

Zaman özellikleri eklendi!


,tarih,istasyon_no,yildirim_sayisi,ort_akim,maks_akim,min_akim,ort_mesafe,yildirim_var,yil,ay,gun,haftanin_gunu,yilin_gunu,mevsim
0,2015-02-06,17285,0,NaN,NaN,NaN,NaN,0,2015,2,6,4,37,0
1,2015-02-06,17920,0,NaN,NaN,NaN,NaN,0,2015,2,6,4,37,0
2,2015-02-06,18234,0,NaN,NaN,NaN,NaN,0,2015,2,6,4,37,0
3,2015-02-06,18771,2,-6.0,-6.0,-6.0,35.5,1,2015,2,6,4,37,0
4,2015-02-06,17815,0,NaN,NaN,NaN,NaN,0,2015,2,6,4,37,0


## 2.7 Verilerin Kaydedilmesi

In [12]:
# Ana veri setini kaydet
ana_df.to_pickle(DATA_PATH + 'ana_veri_seti.pkl')
gunluk_ist_yildirim.to_pickle(DATA_PATH + 'gunluk_istasyon_yildirim.pkl')

print("Veriler kaydedildi!")
print(f"\nAna veri seti: {ana_df.shape}")
print(f"Sütunlar: {ana_df.columns.tolist()}")

Veriler kaydedildi!

Ana veri seti: (23622, 14)
Sütunlar: ['tarih', 'istasyon_no', 'yildirim_sayisi', 'ort_akim', 'maks_akim', 'min_akim', 'ort_mesafe', 'yildirim_var', 'yil', 'ay', 'gun', 'haftanin_gunu', 'yilin_gunu', 'mevsim']


---
**Sonraki Adım:** `03_ozellik_muhendisligi.ipynb` - Özellik Mühendisliği